# M6 · LoRA from scratch

In M5 you fine-tuned every one of Qwen's 494 million weights. It worked, but it was expensive: about 24 GB of memory at the peak, nearly 4 GB of it just for AdamW's notes. A 7-billion-parameter model would need over 100 GB, three times your Mac's memory.

This module builds the idea that fixes that, by hand. **LoRA** (low-rank adaptation) leaves the original weights untouched and trains a tiny add-on next to some of the layers. The add-on is called an **adapter**. Here it will be about a fifth of one percent of the model's size, and it will learn the M5 house style about as well as the full fine-tune did.

By the end of this lesson you'll have:

- built a big matrix out of two thin ones, and counted what that saves
- written a LoRA layer and checked that it starts out exactly like the layer it wraps
- put LoRA layers inside Qwen, frozen everything else, and counted what trains
- trained them on the M5 task, and measured the memory against M5
- merged the adapter back into the weights, so the model is plain Qwen again

**How to work through it** (same as M1 to M5)

- Run each cell from top to bottom with **Shift-Enter**, and read the **What each line does** box under it.
- When a cell says **Predict**, write your guess down before running it.
- New words are **bold** where they're explained, and collected in the glossary in `README.md`.
- Save with **Cmd-S** now and then so the dashboard sees your progress.

This lesson uses the Qwen2.5-0.5B-Instruct model you downloaded in M5, and M5's data. Section 5 trains on the real model and uses up to about 20 GB of memory, so close other big apps first.

## 0. Do we need to change every weight?

Fine-tuning nudges a weight matrix W to a new one. Write the nudge as its own matrix, the **update**: the new weights are W + ΔW (ΔW, "delta W", just means "the change in W").

A full fine-tune learns every number in ΔW. But research on fine-tuning found that the useful part of ΔW is very simple: it can be squeezed into far fewer numbers than W has. So LoRA keeps W **frozen** (never trained, `requires_grad` off) and learns only a squeezed ΔW.

First, your M5 numbers, so we have something to beat.

In [ ]:
import json
import tempfile
import time
from pathlib import Path

import torch
import torch.nn as nn
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL = "Qwen/Qwen2.5-0.5B-Instruct"
SYSTEM = "Rewrite the user's text in plain English. Reply with the rewrite only."
DATA = Path("../m05_full_finetune/data")
device = "mps" if torch.backends.mps.is_available() else "cpu"
torch.manual_seed(0)

m5_run = Path("../../outputs/m05/run_loop.json")
if m5_run.exists():
    m5 = json.loads(m5_run.read_text())
    print(f"Your M5 full fine-tune: held-out loss {m5['heldout_after']:.3f}")
    print(f"  peak memory {m5['peak_mem_gb']} GB, AdamW's state {m5['optimizer_state_gb']} GB")
else:
    print("No M5 run found. The reference M5 full fine-tune: held-out loss 0.94,")
    print("  peak memory about 24 GB, AdamW's state 3.95 GB")

**What each line does**

- The imports are the ones from M5, plus `tempfile` (a throwaway folder, in section 6) and `time`.
- `DATA`: this module reuses M5's 500 jargon and plain English pairs, so the comparison with M5 is fair. `..` means "the folder above", so this is the `m05_full_finetune/data` folder next to this one.
- `m5_run`: the summary `train.py --loop` saved in M5. If it's there, these are your own numbers; if not, the reference run's.
- `torch.manual_seed(0)`: the random numbers in this lesson come out the same every time.

## 1. Low rank: a big matrix from two thin ones

Here's the squeeze. Take a column of 3 numbers and a row of 4, and multiply them with `@`: you get a whole 3 by 4 grid.

In [ ]:
col = torch.tensor([[1.0], [2.0], [3.0]])  # 3 x 1
row = torch.tensor([[10.0, 20.0, 30.0, 40.0]])  # 1 x 4
grid = col @ row
print(grid)
print(grid.numel(), "numbers, made from", col.numel() + row.numel())
print("rank:", torch.linalg.matrix_rank(grid).item())

**What each line does**

- `col @ row`: a 3 x 1 matrix times a 1 x 4 matrix is 3 x 4 (M1). Each entry is one number from `col` times one from `row`.
- Every row of the grid is the same row, `[10, 20, 30, 40]`, times 1, 2 or 3. So 12 numbers are described by 7.
- `torch.linalg.matrix_rank`: how many truly different directions a matrix holds. This one has just 1.

A matrix built like this is **low-rank**. The **rank** `r` is how many column-times-row pieces you add up: with `r` columns side by side (a matrix B, out x r) and `r` rows stacked (a matrix A, r x in), `B @ A` is a full out x in matrix built from only `r * (out + in)` numbers.

That's LoRA's update: **ΔW = B @ A**.

Qwen's `q_proj` layer (one of the attention projections from M2 section 3) is a 896 by 896 weight matrix: 802,816 numbers.

**Predict:** with rank 8, how many numbers are in B and A together?

In [ ]:
r = 8
B = torch.randn(896, r)
A = torch.randn(r, 896)
update = B @ A
print(tuple(update.shape), f"{update.numel():,} numbers in the update")
print(f"{B.numel() + A.numel():,} numbers in B and A: r * (896 + 896) = {r * (896 + 896):,}")
print(f"that's {100 * (B.numel() + A.numel()) / update.numel():.1f}% of the full matrix")
print("rank:", torch.linalg.matrix_rank(update).item())

**What each line does**

- `B` is 896 x 8 and `A` is 8 x 896, so `B @ A` is 896 x 896: exactly the shape of the weight it will be added to.
- `r * (896 + 896)` = 14,336 numbers instead of 802,816: under 2%. That sum is exercise 2.
- The rank is 8: however big the matrix, it holds only 8 directions. The bet LoRA makes is that 8 directions are enough to learn a style.

## 2. A LoRA layer, by hand

A LoRA layer wraps an existing `nn.Linear` (call it `base`) and adds the update to what it computes:

**output = base(x) + scale · (x @ Aᵀ @ Bᵀ)**

That's the same as using the weight W + scale · B @ A, without ever building that big matrix. (`.T` flips a matrix; it's there because `nn.Linear` stores its weight as out x in.)

How A and B start matters:

- **B starts at zero.** Then B @ A is zero, and the layer gives *exactly* what `base` gives. Training starts from the original model, not from a damaged one.
- **A starts random.** You'll see why in a moment.

Let's build one on a tiny 6-in, 4-out layer.

In [ ]:
base = nn.Linear(6, 4)
for p in base.parameters():
    p.requires_grad_(False)

r, alpha = 2, 4
A = nn.Parameter(torch.randn(r, 6) / 6**0.5)
B = nn.Parameter(torch.zeros(4, r))
scale = alpha / r

x = torch.randn(3, 6)
out = base(x) + scale * (x @ A.T @ B.T)
print("same as the base layer:", torch.equal(out, base(x)))

**What each line does**

- `p.requires_grad_(False)`: freezes the base layer's weight and bias (M1 section 2). `.backward()` will never compute gradients for them, so the optimizer can't change them.
- `nn.Parameter(...)`: marks a tensor as a weight to train. `model.parameters()` finds it, and it has `requires_grad` on.
- `torch.randn(r, 6) / 6**0.5`: random numbers, shrunk by the square root of the input size so the result isn't huge. `torch.zeros(4, r)`: all zeros.
- `scale = alpha / r`: section 3 explains it.
- `torch.equal`: `True` only if every number matches exactly. B is zero, so the update adds nothing.

**Predict:** after a `.backward()`, which of B, A and the base weight get a gradient that isn't zero?

In [ ]:
out.pow(2).sum().backward()
print("B's gradient:   ", B.grad.abs().sum().item())
print("A's gradient:   ", A.grad.abs().sum().item())
print("base's gradient:", base.weight.grad)

A0 = nn.Parameter(torch.zeros(r, 6))
B0 = nn.Parameter(torch.zeros(4, r))
(base(x) + scale * (x @ A0.T @ B0.T)).pow(2).sum().backward()
print("both zero:", A0.grad.abs().sum().item(), B0.grad.abs().sum().item())

**What each line does**

- `out.pow(2).sum()`: a made-up loss, just to have something to call `.backward()` on.
- B's gradient isn't zero: it depends on A (which is random) and x. So the first step moves B away from zero.
- A's gradient *is* zero for now: it depends on B, which is still zero. After one step B isn't zero any more, and A starts learning too.
- `None` for the base: frozen weights get no gradient at all.
- With A and B *both* zero, every gradient is zero, forever: nothing could ever train. That's why A starts random.

That's the whole layer. Wrapped up as an `nn.Module` (M1 section 4) so it can stand in for an `nn.Linear` anywhere, it looks like this. It's exercise 1, so read it closely.

In [ ]:
class LoRALinear(nn.Module):
    def __init__(self, base, r, alpha):
        super().__init__()
        self.base = base
        for p in self.base.parameters():
            p.requires_grad_(False)
        w = base.weight
        self.A = nn.Parameter(
            torch.randn(r, base.in_features, device=w.device, dtype=w.dtype) / base.in_features**0.5
        )
        self.B = nn.Parameter(torch.zeros(base.out_features, r, device=w.device, dtype=w.dtype))
        self.scale = alpha / r

    def forward(self, x):
        return self.base(x) + self.scale * (x @ self.A.T @ self.B.T)


layer = LoRALinear(nn.Linear(6, 4), r=2, alpha=4)
print(layer)
print([(name, tuple(p.shape), p.requires_grad) for name, p in layer.named_parameters()])

**What each line does**

- `self.base = base`: keeps the original layer inside the new one. Its weights stay where they are.
- `base.in_features`, `base.out_features`: an `nn.Linear` knows its own sizes (6 and 4 here).
- `device=w.device, dtype=w.dtype`: make A and B wherever the base weight lives (the GPU, for the real model) and in the same number format. Without this, A and B would be on the CPU while the model is on the GPU, and the multiply would fail.
- `forward`: the formula from the top of this section.
- `named_parameters()`: A and B train; `base.weight` and `base.bias` don't.

## 3. The scale: alpha / r

The update is multiplied by **scale = alpha / r** before it's added. **alpha** is a number you choose. Think of the scale as a volume knob on the adapter: the same A and B, turned up, change the layer more.

Let's pretend training has moved B a little, then turn the knob.

In [ ]:
with torch.no_grad():
    layer.B.fill_(0.1)
x = torch.randn(3, 6)
for alpha in [2, 4, 8]:
    layer.scale = alpha / 2
    change = (layer(x) - layer.base(x)).abs().mean().item()
    print(f"alpha {alpha}: scale {layer.scale}, the output moves by {change:.3f} on average")

**What each line does**

- `torch.no_grad()` and `.fill_(0.1)`: set every number in B to 0.1 by hand, without recording it for gradients.
- `layer(x) - layer.base(x)`: how much the adapter changes the output.
- Double alpha and the change doubles. The scale is a plain multiplier.

Why divide by `r` at all? In LoRA's original paper, alpha stays fixed while you try different ranks. Dividing by `r` keeps the size of the update roughly steady as the rank changes, so you don't have to re-tune the learning rate for each rank. Another common habit, used by `train.py`, is **alpha = 2r**: then the `r`s cancel out, the scale is always 2, and the learning rate is the one thing you tune. Here r = 8 and alpha = 16.

## 4. Putting LoRA inside Qwen

Now the real model. First the tools from M5: load Qwen, format prompts, and measure the loss on the 50 validation pairs.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForCausalLM.from_pretrained(MODEL, dtype=torch.float32).to(device)
pad_id = tokenizer.pad_token_id
END = tokenizer.convert_tokens_to_ids("<|im_end|>")


def read_jsonl(path):
    return [json.loads(line) for line in open(path)]


train_rows = read_jsonl(DATA / "train.jsonl")
val_rows = read_jsonl(DATA / "val.jsonl")
test_rows = read_jsonl(DATA / "test.jsonl")


def prompt_ids(text, system=SYSTEM):
    messages = [{"role": "user", "content": text}]
    if system is not None:
        messages.insert(0, {"role": "system", "content": system})
    return tokenizer.apply_chat_template(messages, add_generation_prompt=True)["input_ids"]


def answer_ids(text):
    return tokenizer.encode(text, add_special_tokens=False) + [END]


def make_batch(rows):
    examples = [(prompt_ids(r["jargon"]), answer_ids(r["plain"])) for r in rows]
    longest = max(len(p) + len(a) for p, a in examples)
    ids, mask, labs = [], [], []
    for p, a in examples:
        pad = longest - len(p) - len(a)
        ids.append(p + a + [pad_id] * pad)
        mask.append([1] * (len(p) + len(a)) + [0] * pad)
        labs.append([-100] * len(p) + a + [-100] * pad)
    return [torch.tensor(x, dtype=torch.long, device=device) for x in (ids, mask, labs)]


@torch.no_grad()
def val_loss():
    model.eval()
    losses = []
    for i in range(0, len(val_rows), 10):
        ids, mask, labs = make_batch(val_rows[i : i + 10])
        losses.append(model(input_ids=ids, attention_mask=mask, labels=labs).loss.item())
    model.train()
    return sum(losses) / len(losses)


@torch.no_grad()
def reply(text, system=SYSTEM, max_new=96):
    model.eval()
    ids = torch.tensor([prompt_ids(text, system)], device=device)
    out = model.generate(ids, max_new_tokens=max_new, do_sample=False)
    model.train()
    return tokenizer.decode(out[0, ids.shape[1] :], skip_special_tokens=True).strip()


before = val_loss()
print(f"{sum(p.numel() for p in model.parameters()):,} parameters, on {device}")
print("val loss of the original model:", round(before, 3))

**What each line does**

- Everything here is from M5 sections 1 to 5: the model in float32, the chat template, padding, -100 labels on the prompt and the padding, and the validation loss.
- `make_batch` is your M5 `build_example` and `collate` in one function; `train.py` uses your own M5 code instead.
- `before`: the original model's validation loss, about 1.55, the same number M5 started from.

Where do the LoRA layers go? Print one of Qwen's 24 layers' attention block.

In [ ]:
print(model.model.layers[0].self_attn)
names = [name for name, module in model.named_modules() if name.endswith("q_proj")]
print(len(names), "q_proj layers, the first is", repr(names[0]))

**What each line does**

- `model.model.layers[0].self_attn`: the attention part of the first block (M2 sections 2 and 3). Its four `nn.Linear` layers make the queries, keys, values and the output.
- `k_proj` and `v_proj` are 896 to 128: Qwen shares keys and values between groups of heads, so they're smaller.
- `named_modules()`: every layer in the model with its dotted name, like a file path: `model.layers.0.self_attn.q_proj`.

The layers we put adapters on are the **target modules**. LoRA's original paper mostly adapted just two of the attention projections, the queries and the values. Using all four, as we do (`q_proj`, `k_proj`, `v_proj` and `o_proj`), is a common choice today.

To swap a layer, you need its parent (the `self_attn` block) and its own short name. `rpartition(".")` splits a dotted name at its last dot, `model.get_submodule` finds the parent, and `setattr` puts a new layer in place of the old one.

We freeze the whole model first, so that only the adapters will train.

In [ ]:
for p in model.parameters():
    p.requires_grad_(False)

name = names[0]
parent_name, _, child_name = name.rpartition(".")
print(repr(parent_name), repr(child_name))
parent = model.get_submodule(parent_name)
old = getattr(parent, child_name)
setattr(parent, child_name, LoRALinear(old, r=8, alpha=16))
print(model.get_submodule(name))

**What each line does**

- The `for` loop freezes all 494 million weights.
- `name.rpartition(".")`: three parts, everything before the last dot, the dot, and everything after: `"model.layers.0.self_attn"`, `"."`, `"q_proj"`. The `_` takes the dot, which we don't need.
- `getattr(parent, child_name)`: the same as `parent.q_proj`, with the name as a string. `setattr(parent, child_name, new)` is the same as `parent.q_proj = new`.
- The printout shows the swap worked: a `LoRALinear` with the original `Linear` inside it.

Now all of them. That loop is exercise 3.

**Predict:** how many layers will be replaced? And how many trainable numbers will that be? (Section 1 has the count for one `q_proj`; `k_proj` and `v_proj` are 896 in, 128 out.)

In [ ]:
targets = ["q_proj", "k_proj", "v_proj", "o_proj"]
replaced = []
for name, module in list(model.named_modules()):
    parent_name, _, child_name = name.rpartition(".")
    if child_name in targets and isinstance(module, nn.Linear):
        setattr(model.get_submodule(parent_name), child_name, LoRALinear(module, r=8, alpha=16))
        replaced.append(name)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(len(replaced) + 1, "layers have an adapter")
print(f"{trainable:,} of {total:,} parameters train ({100 * trainable / total:.2f}%)")
print("val loss now:", round(val_loss(), 3), "(before:", round(before, 3), ")")

**What each line does**

- `list(model.named_modules())`: a copy of the list of layers, made first. Changing a model while walking through it directly can skip or repeat layers.
- `isinstance(module, nn.Linear)`: only plain linear layers. The `q_proj` swapped by hand above is already a `LoRALinear`, so it's skipped, and that's why the count adds 1. This check also stops a second run of the loop from wrapping a LoRA layer in another one.
- `p.requires_grad`: `True` only for the new A and B matrices. Counting them is exercise 4.
- 96 layers (24 blocks x 4), 1,081,344 trainable numbers: 0.22% of the model. These are the **trainable parameters**.
- The validation loss hasn't moved: every B is still zero, so the model is exactly the original.

## 5. Training only the adapters

The same loop as M5 section 5, with two changes:

- The optimizer is given only the trainable parameters.
- The learning rate is 2e-4, twenty times M5's. The adapters start from nothing, so they need bigger steps; the frozen weights can't be damaged by them.

**Predict:** AdamW keeps 8 bytes of notes per trainable number. How big will its state be? And will the peak memory fall as much as that?

In [ ]:
optimizer = torch.optim.AdamW(
    [p for p in model.parameters() if p.requires_grad], lr=2e-4, weight_decay=0.0
)
if device == "mps":
    torch.mps.empty_cache()
peak = 0.0
order = torch.randperm(len(train_rows), generator=torch.Generator().manual_seed(0)).tolist()
start = time.perf_counter()
for step, i in enumerate(range(0, len(train_rows), 8)):
    ids, mask, labs = make_batch([train_rows[j] for j in order[i : i + 8]])
    loss = model(input_ids=ids, attention_mask=mask, labels=labs).loss
    optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], 1.0)
    optimizer.step()
    if device == "mps":
        peak = max(peak, torch.mps.driver_allocated_memory() / 1e9)
    if step % 10 == 0:
        print(f"  step {step:3d}  loss {loss.item():.3f}")
print(f"one epoch in {time.perf_counter() - start:.0f}s")

state = sum(
    v.numel() * v.element_size()
    for entry in optimizer.state.values()
    for v in entry.values()
    if torch.is_tensor(v)
)
print(f"AdamW's state: {state / 1e6:.1f} MB")
if device == "mps":
    print(f"peak memory: {peak:.1f} GB")
after = val_loss()
print("val loss:", round(before, 3), "->", round(after, 3))

**What each line does**

- `[p for p in model.parameters() if p.requires_grad]`: only the A and B matrices. AdamW keeps notes only for what it's given.
- `torch.mps.empty_cache()`: hands back memory the GPU was keeping from earlier cells, so the peak we measure is mostly this training.
- The loop is M5's: batch, loss, `zero_grad`, `backward`, clip, `step`. The 494 million frozen weights are used in every forward and backward pass, but never changed.
- AdamW's state is the sum from your M5 exercise 4: about 8.7 MB, against M5's 3.95 GB.
- Peak memory falls far less than that. The frozen weights (2 GB) are still loaded, and the **activations** the backward pass needs are still saved: the gradients have to flow back through every frozen layer to reach the adapters underneath. In the reference run this cell peaked at about 14 GB. `train.py` measures memory the same way M5's did (including what the GPU keeps cached between steps), and there LoRA peaked at about 20 GB against the full fine-tune's 24.
- After one epoch the validation loss (about 0.90 in the reference run) is already as low as M5's 2-epoch full fine-tune reached. `train.py` does 2 epochs with a learning-rate schedule.

In [ ]:
for row in test_rows[:3]:
    print("jargon:   ", row["jargon"])
    print("LoRA:     ", reply(row["jargon"]))
    print("reference:", row["plain"])
    print()

**What each line does**

- Three test sentences the model never trained on, rewritten by the model with its adapters.
- Compare them with what the original model wrote in M5 section 1: shorter, plainer and in the house style, after training 0.22% of the weights.
- Read them closely, though. In the reference run the knee-replacement rewrite *invented* a price that isn't in the sentence. A low loss doesn't promise every output is faithful, which is why M11 is all about evaluation. (`train.py`'s 2-epoch run rewrote that one without the made-up price.)

## 6. Merging the adapter back

The adapter is an extra multiply in 96 places. When you're done training, you can fold it into the weights for good: W becomes W + scale · B @ A, one plain `nn.Linear` again. That's **merging**.

**Predict:** will the merged layer give exactly the same outputs as the LoRA layer?

In [ ]:
layer = model.get_submodule("model.layers.0.self_attn.q_proj")
base = layer.base
merged = nn.Linear(base.in_features, base.out_features, bias=base.bias is not None, device=device)
with torch.no_grad():
    merged.weight.copy_(base.weight + layer.scale * layer.B @ layer.A)
    if base.bias is not None:
        merged.bias.copy_(base.bias)

x = torch.randn(2, 5, 896, device=device)
print("biggest difference:", (merged(x) - layer(x)).abs().max().item())

**What each line does**

- `nn.Linear(..., bias=base.bias is not None, device=device)`: a fresh layer of the same size, on the GPU. `bias=` is `True` here, since Qwen's `q_proj` has a bias.
- `layer.B @ layer.A`: the 896 x 896 update, built once. Times the scale, plus the frozen W.
- `.copy_(...)`: fills the new layer's weight in place. Inside `torch.no_grad()`, since this isn't training.
- `if base.bias is not None`: `q_proj` has a bias to copy, but `o_proj` doesn't (you saw `bias=False` in section 4).
- `layer.base` itself is left alone: the merged layer is a fresh one. Changing the base weight in place would make the LoRA layer add the update twice.
- The difference is tiny (about 0.00005 in the reference run): a rounding difference, because the numbers are added up in a different order. It's the same layer, written as one matrix. That's exercise 5.

Before merging everything, save the adapter on its own. It's just the A and B matrices.

In [ ]:
adapter = {n: p.detach().cpu() for n, p in model.named_parameters() if p.requires_grad}
with tempfile.TemporaryDirectory() as tmp:
    torch.save(adapter, Path(tmp) / "adapter.pt")
    size = (Path(tmp) / "adapter.pt").stat().st_size
print(len(adapter), "tensors,", f"{size / 1e6:.1f} MB")
print(f"the whole model in float32: {4 * total / 1e9:.1f} GB")

for name, module in list(model.named_modules()):
    if isinstance(module, LoRALinear):
        parent_name, _, child_name = name.rpartition(".")
        b = module.base
        new = nn.Linear(b.in_features, b.out_features, bias=b.bias is not None, device=device)
        with torch.no_grad():
            new.weight.copy_(b.weight + module.scale * module.B @ module.A)
            if b.bias is not None:
                new.bias.copy_(b.bias)
        setattr(model.get_submodule(parent_name), child_name, new)

print(model.model.layers[0].self_attn.q_proj)
print("val loss after merging:", round(val_loss(), 3), "(before merging:", round(after, 3), ")")

**What each line does**

- `p.detach().cpu()`: a copy of each A and B on the CPU, cut off from the gradient bookkeeping, ready to save.
- `torch.save(...)` into a `TemporaryDirectory`: written to a throwaway folder that's deleted when the `with` block ends, just to measure the size.
- 192 tensors (an A and a B for each of 96 layers), about 4.4 MB. The model they adapt is 2 GB. You could keep a different 4 MB adapter for every task, and share one base model.
- The loop merges every LoRA layer and puts the plain layer back. `if b.bias is not None`: `o_proj` has no bias, so there's nothing to copy for it. Afterwards the model is ordinary Qwen again: same layers, same speed, no adapter code needed to run it.
- The validation loss is unchanged. Once merged, though, the adapter can't be switched off or swapped. That's why you'd keep the saved file.

## 7. Which layers, and how big a rank?

Two choices: which layers to target, and the rank. These are the reference results from `train.py` (2 epochs, learning rate 2e-4, alpha = 2r), against M5's full fine-tune:

| Run | Trainable | Held-out loss | Adapter file | AdamW's state |
|---|---|---|---|---|
| Full fine-tune (M5) | 100% | 0.94 | 2 GB (the whole model) | 3.95 GB |
| rank 2, attention | 0.05% | 0.940 | 1.1 MB | 2.2 MB |
| **rank 8, attention (the default)** | **0.22%** | **0.903** | **4.4 MB** | **8.7 MB** |
| rank 32, attention | 0.88% | 0.907 | 17.4 MB | 34.6 MB |
| rank 8, attention + MLP | 0.89% | 0.913 | 17.7 MB | 35.2 MB |

(The held-out loss is measured on the 50 validation pairs, with the reference rewrites; lower is better.)

What they show:

- **Even rank 2 matches the full fine-tune**, with one two-thousandth of the trainable weights. A house style really is a small change.
- **Bigger isn't automatically better.** Rank 32, or adding the MLP layers (`gate_proj`, `up_proj`, `down_proj`), trains four times as much and does no better here. On harder tasks, like teaching new knowledge, the bigger options help more.
- **Peak memory barely changes** between these runs (about 20 GB each): it's the frozen model and the activations, not the adapter. M8 attacks that part, by storing the frozen weights in 4 bits.

Finally, the forgetting check from M5 section 8. The original weights are untouched, and only the adapter changed them a little.

In [ ]:
general = read_jsonl(DATA / "general.jsonl")
right = 0
for g in general:
    answer = reply(g["question"], system=None)
    right += g["expect"] in answer.lower()
print(right, "of", len(general), "ordinary questions right")

**What each line does**

- The same 10 questions as M5, with no system message, on the merged model.
- In the reference runs, the default LoRA run answered all 10 right, like the original model. LoRA doesn't make forgetting impossible (some of the bigger runs above got 9 of 10), but changing fewer weights tends to forget less.

## 8. Your task

You've seen every piece. Time to write them yourself.

1. Open `exercises.py`. There are five exercises, each with plain words and numbered steps, and each points to the section above that shows what you need:
   1. `LoRALinear` (section 2)
   2. `lora_params` (section 1)
   3. `inject_lora` (section 4)
   4. `count_trainable` (section 4)
   5. `merge_lora` (section 6)
2. Check your work as you go: `make check M=06`. The run tests fail until step 4.
3. Answer the six questions in `answers.toml`. Predict first, then check yourself in this notebook.
4. Train (close this notebook first, or restart its kernel, to free its memory):
   - `uv run python modules/m06_lora_from_scratch/train.py`

   It puts your LoRA layers in Qwen, trains them for 2 epochs with your M5 loop pieces, merges them with your `merge_lora`, and scores the result. About 3 minutes. To pass, under 1% of the weights may train, merging must not change the held-out loss, and the loss must come within 5% of your M5 full fine-tune's (or the reference 0.94, if yours did better).
5. Run `make check M=06` again. When everything passes, M6 turns green on the dashboard.

Stuck? `make hint M=06 EX=3` gives a nudge for exercise 3. Every term is in the glossary in `README.md`.

**Stretch:** run `train.py --r 2`, `--r 32` and `--targets all`. Each run is saved in `outputs/m06/sweep/`. Plot held-out loss against the number of trainable parameters (matplotlib, from M1), and see whether your results match the table in section 7.